In [ ]:
 python -m benchmark_exp.Run_Detector_U --save True --AD_Name Chronos2Fast  

In [8]:
import pandas as pd

# пути к файлам
model1_path = "../eval/metrics/uni/Chronos2Fast.csv"
model2_path = "../eval/metrics/uni/Sub_PCA.csv"

m1 = pd.read_csv(model1_path)
m2 = pd.read_csv(model2_path)

metrics = [
    "AUC-PR",
    "AUC-ROC",
    "VUS-PR",
    "VUS-ROC",
    "Standard-F1",
    "PA-F1",
    "Event-based-F1",
    "R-based-F1",
    "Affiliation-F"
]

In [5]:
# --- общее сравнение моделей ---

summary = pd.DataFrame({
    "model1_mean": m1[metrics].mean(),
    "model2_mean": m2[metrics].mean(),
    "model1_median": m1[metrics].median(),
    "model2_median": m2[metrics].median()
})

print("\n=== GLOBAL METRIC COMPARISON ===")
summary


=== GLOBAL METRIC COMPARISON ===


,model1_mean,model2_mean,model1_median,model2_median
AUC-PR,0.649503,0.023343,0.564920,0.007866
AUC-ROC,0.946327,0.549562,0.998211,0.548117
VUS-PR,0.819973,0.124164,0.843688,0.062994
VUS-ROC,0.946106,0.593260,0.999324,0.573715
Standard-F1,0.805345,0.053567,0.799995,0.021505
PA-F1,0.803504,0.049807,0.800000,0.020240
Event-based-F1,0.803504,0.049807,0.800000,0.020240
R-based-F1,0.830545,0.122773,0.845070,0.098246
Affiliation-F,0.958277,0.728676,0.998846,0.694007


In [6]:
# --- file-wise сравнение ---

df = m1.merge(m2, on="file", suffixes=("_m1", "_m2"))

winners = {}

for m in metrics:
    
    better_m1 = (df[f"{m}_m1"] > df[f"{m}_m2"]).sum()
    better_m2 = (df[f"{m}_m1"] < df[f"{m}_m2"]).sum()
    ties = (df[f"{m}_m1"] == df[f"{m}_m2"]).sum()
    
    winners[m] = {
        "model1_wins": better_m1,
        "model2_wins": better_m2,
        "ties": ties
    }

wins_df = pd.DataFrame(winners).T

print("\n=== FILE-WISE WINS ===")
wins_df



=== FILE-WISE WINS ===


,model1_wins,model2_wins,ties
AUC-PR,161,0,1
AUC-ROC,154,7,1
VUS-PR,160,1,1
VUS-ROC,152,9,1
Standard-F1,161,0,1
PA-F1,161,0,1
Event-based-F1,161,0,1
R-based-F1,161,0,1
Affiliation-F,154,7,1


In [7]:
# --- детальное сравнение по каждому файлу ---

file_results = []

for _, row in df.iterrows():
    
    file_name = row["file"]
    
    row_result = {"file": file_name}
    
    for m in metrics:
        
        v1 = row[f"{m}_m1"]
        v2 = row[f"{m}_m2"]
        
        if v1 > v2:
            winner = "model1"
        elif v2 > v1:
            winner = "model2"
        else:
            winner = "tie"
        
        row_result[m] = winner
    
    file_results.append(row_result)

file_comparison = pd.DataFrame(file_results)

print("\n=== FILE-WISE WINNER TABLE ===")
file_comparison


=== FILE-WISE WINNER TABLE ===


,file,AUC-PR,AUC-ROC,VUS-PR,VUS-ROC,Standard-F1,PA-F1,Event-based-F1,R-based-F1,Affiliation-F
0,633_YAHOO_id_83_WebService_tr_500_1st_223.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
1,710_YAHOO_id_160_WebService_tr_500_1st_253.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
2,758_YAHOO_id_208_WebService_tr_500_1st_24.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
3,760_YAHOO_id_210_WebService_tr_500_1st_23.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
4,623_YAHOO_id_73_WebService_tr_500_1st_117.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
...,...,...,...,...,...,...,...,...,...,...
157,754_YAHOO_id_204_WebService_tr_500_1st_28.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
158,691_YAHOO_id_141_WebService_tr_500_1st_57.csv,model1,model2,model1,model2,model1,model1,model1,model1,model2
159,698_YAHOO_id_148_WebService_tr_500_1st_186.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1
160,645_YAHOO_id_95_WebService_tr_500_1st_1047.csv,model1,model1,model1,model1,model1,model1,model1,model1,model1


In [9]:
# --- на каком файле модель сильно выигрывает ---

diff_df = df.copy()

for m in metrics:
    diff_df[m + "_diff"] = diff_df[f"{m}_m1"] - diff_df[f"{m}_m2"]

print("\n=== LARGEST DIFFERENCES ===")

for m in metrics:
    
    idx = diff_df[m + "_diff"].abs().idxmax()
    
    print(
        m,
        diff_df.loc[idx, "file"],
        diff_df.loc[idx, m + "_diff"]
    )


=== LARGEST DIFFERENCES ===
AUC-PR 610_YAHOO_id_60_WebService_tr_500_1st_878.csv 0.9992716678805535
AUC-ROC 696_YAHOO_id_146_Synthetic_tr_500_1st_893.csv 0.947887323943662
VUS-PR 788_YAHOO_id_238_WebService_tr_500_1st_973.csv 0.9990083876770023
VUS-ROC 574_YAHOO_id_24_WebService_tr_500_1st_202.csv 0.917216331590721
Standard-F1 610_YAHOO_id_60_WebService_tr_500_1st_878.csv 0.99853941049461
PA-F1 610_YAHOO_id_60_WebService_tr_500_1st_878.csv 0.9985454545454546
Event-based-F1 610_YAHOO_id_60_WebService_tr_500_1st_878.csv 0.9985454545454542
R-based-F1 696_YAHOO_id_146_Synthetic_tr_500_1st_893.csv 0.9989077007099946
Affiliation-F 696_YAHOO_id_146_Synthetic_tr_500_1st_893.csv 0.3336267838353718
